# Qwen3.5-4B LoRA Jacobian-Lens smoke test (T4)

Fits one generic WikiText prompt with the merged prompt-injection-resistant LoRA. It uses all source layers L0-L30, target layer L31, `max_seq_len=128`, and `dim_batch=1`. The base model is downloaded from Hugging Face into the temporary Colab disk. No Google Drive or project clone is required.

Before running, select a **T4 GPU**. In the upload cell, select only `adapter_config.json` and `adapter_model.safetensors` from `outputs/qwen35-4b-lora-pi-r8-stage1-120/` on the Mac.

In [ ]:
!pip install -q --upgrade \
    "transformers==5.14.1" \
    "datasets==5.0.0" \
    "accelerate==1.14.0" \
    "peft==0.19.1" \
    "torchao>=0.16.0"
!pip install -q \
    git+https://github.com/anthropics/jacobian-lens.git@581d398613e5602a5af361e1c34d3a92ea82ba8e


In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import psutil
import torch

assert torch.cuda.is_available(), "No CUDA GPU: select a GPU runtime first."
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", gpu_name)
print(f"VRAM: {total_vram:.2f} GiB")
print(f"System RAM: {psutil.virtual_memory().total / 1024**3:.2f} GiB")
!df -h /content
assert "T4" in gpu_name, f"This smoke test is intended for a T4, got {gpu_name}"


## Upload the 62 MB LoRA adapter

Select exactly these two local files:

- `outputs/qwen35-4b-lora-pi-r8-stage1-120/adapter_config.json`
- `outputs/qwen35-4b-lora-pi-r8-stage1-120/adapter_model.safetensors`

In [ ]:
from google.colab import files

uploaded = files.upload()
required = {"adapter_config.json", "adapter_model.safetensors"}
assert set(uploaded) == required, f"Expected {required}, got {set(uploaded)}"
adapter_dir = Path("/content/qwen35-4b-lora-pi-r8-stage1-120")
adapter_dir.mkdir(parents=True, exist_ok=True)
for filename, payload in uploaded.items():
    (adapter_dir / filename).write_bytes(payload)
del uploaded
gc.collect()
print("Adapter ready:", adapter_dir)


## Download Qwen, merge the LoRA, and wrap it for jlens

The Hugging Face cache is under `/content/hf-cache` and disappears with the runtime. FP16 is used on the T4.

In [ ]:
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import jlens
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-4B"
adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text())
assert adapter_config["base_model_name_or_path"] == MODEL_ID

gc.collect()
torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, low_cpu_mem_usage=True
).to("cuda")
peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = peft_model.merge_and_unload(safe_merge=True)
model = jlens.from_hf(merged_model, tokenizer)
assert model.n_layers == 32 and model.d_model == 2560

free_vram, total_vram_bytes = torch.cuda.mem_get_info()
print(model)
print(f"VRAM allocated after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"VRAM reserved after load:  {torch.cuda.memory_reserved() / 1024**3:.2f} GiB")
print(f"VRAM free after load:      {free_vram / 1024**3:.2f} GiB")
print(f"System RAM available:      {psutil.virtual_memory().available / 1024**3:.2f} GiB")


## Real one-prompt all-layer fit

This is the actual smoke test, not only a synthetic allocation. On the previously tested T4 it took about 28 minutes.

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=1)
source_layers = list(range(31))
checkpoint_path = Path("/content/qwen35_lora_t4_smoke_checkpoint.pt")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
smoke_lens = jlens.fit(
    model,
    prompts,
    source_layers=source_layers,
    target_layer=31,
    dim_batch=1,
    max_seq_len=128,
    skip_first=16,
    checkpoint_path=str(checkpoint_path),
    checkpoint_every=1,
    resume=False,
)
fit_seconds = time.perf_counter() - started
free_vram, total_vram_bytes = torch.cuda.mem_get_info()
print(f"Fit time: {fit_seconds / 60:.2f} minutes")
print(f"Peak allocated VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
print(f"VRAM free after fit: {free_vram / 1024**3:.2f} GiB")
print(f"System RAM available: {psutil.virtual_memory().available / 1024**3:.2f} GiB")
print(smoke_lens)


In [ ]:
lens_path = Path("/content/qwen35_lora_t4_smoke_lens_n1.pt")
report_path = Path("/content/qwen35_lora_t4_smoke_report.json")
smoke_lens.save(str(lens_path), dtype=torch.float16)
report = {
    "model_id": MODEL_ID,
    "adapter": adapter_dir.name,
    "gpu": gpu_name,
    "source_layers": source_layers,
    "target_layer": 31,
    "n_prompts": smoke_lens.n_prompts,
    "dim_batch": 1,
    "max_seq_len": 128,
    "skip_first": 16,
    "fit_seconds": fit_seconds,
    "peak_allocated_vram_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "checkpoint_bytes": checkpoint_path.stat().st_size,
    "lens_bytes": lens_path.stat().st_size,
}
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))


## Download results to the Mac

Download the report and lens. The checkpoint is optional for this one-prompt smoke test and is much larger.

In [ ]:
files.download(str(report_path))
files.download(str(lens_path))
# Optional (~800 MB):
# files.download(str(checkpoint_path))
